# AgriBot RL Analysis — SAC vs PPO for Fruit Reaching

The `FruitReachEnv` task: move the UR5e tool-center-point to a randomly
placed fruit. Continuous 6-DoF control, dense shaped reward, success = TCP
within 6 cm. We compare **SAC** (off-policy) and **PPO** (on-policy) at an
equal training budget, then discuss when a *learned policy* beats a
*classical planner* (MoveIt/IK) for harvesting.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath('..'))

RUNS = os.path.join('..', 'training', 'runs')
def load(run):
    d = np.load(os.path.join(RUNS, run, 'evaluations.npz'))
    return d['timesteps'], d['results'].mean(1), d['successes'].mean(1)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for run, color in [('sac_seed0_reach', 'C0'), ('ppo_seed0_reach', 'C1')]:
    try:
        ts, rew, succ = load(run)
    except FileNotFoundError:
        print('missing', run); continue
    ax1.plot(ts, rew, color=color, marker='o', ms=3, label=run.split('_')[0].upper())
    ax2.plot(ts, succ*100, color=color, marker='o', ms=3, label=run.split('_')[0].upper())
ax1.set(xlabel='timesteps', ylabel='mean eval return', title='Return')
ax2.set(xlabel='timesteps', ylabel='success rate (%)', title='Reach success', ylim=(0,100))
for ax in (ax1, ax2): ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); plt.show()

## Result

At equal budget (~300k steps) **SAC reaches ~100% success while PPO reaches
~45%**. SAC's replay buffer reuses every transition many times, so it is far
more sample-efficient on this well-shaped continuous-control task; PPO's
on-policy updates need many more environment interactions to catch up.

## Learned policy vs. classical planner — when to use which?

The orchard demo (`orchard/`) reaches fruit with **damped-least-squares IK** —
a classical planner. The RL policy here reaches fruit with a **learned
controller**. Both solve "move the TCP to a point," so which is right?

| | Classical IK / MoveIt | Learned RL policy |
|---|---|---|
| Needs a model of the arm | yes (exact) | no (learns dynamics) |
| Guarantees / interpretability | strong | weak (black box) |
| Contact-rich / deformable fruit | brittle | can learn compliance |
| Reacts to a moving target | replan each step | reactive by construction |
| Training cost | none | hours of simulation |

**Takeaway for harvesting:** use the planner for the free-space reach (it is
exact, safe, and needs no training), and reserve RL for the parts a planner
handles badly — compliant grasping of soft fruit, or reactive final-approach
under a wobbling branch. A promising hybrid is *behaviour cloning from planner
demonstrations, then RL fine-tuning* — the best of both.